# Feature Engineering

In the previous notebook we set a baseline using CatBoost model and achieved an `OOF ROC-AUC` of ***0.946094***

*Goal* of current notebook which new features provide improvement to ROC-AUC and which provide little or no additional improvement.

## Imports and Config

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

SEED = 42
N_SPLITS = 5
BASELINE_AUC = 0.946094

MODEL_PARAMS = {
    "iterations": 3000,
    "depth": 6,
    "learning_rate": 0.05,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": SEED,
    "verbose": False,
    "thread_count": -1,
}

## Load Data

In [3]:
root = Path("../playground-series-s6e11")

train_df = pd.read_csv(root / "train.csv")

TARGET = "is_fraud"

base_features = [
    col for col in train_df.columns
    if col not in ["id", TARGET]
]

print("Train shape:", train_df.shape)
print("Baseline features:", len(base_features))

Train shape: (699960, 26)
Baseline features: 24


## Reusable CV function

We will use the same `CatBoost` configuration and 5-fold cross-validation as in our baseline.

To make experiments easier, we define a reusable function that trains the model, generates OOF predictions, and calculates ROC-AUC.

In [4]:
def evaluate_features(df, features, target=TARGET):
    X = df[features].copy()
    y = df[target]

    # Match the baseline preprocessing
    bool_cols = X.select_dtypes(include="bool").columns
    X[bool_cols] = X[bool_cols].astype(int)

    cat_features = X.select_dtypes(
        include=["object", "category", "str"]
    ).columns.tolist()

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    )

    oof_pred = np.zeros(len(df))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X, y), start=1
    ):
        model = CatBoostClassifier(**MODEL_PARAMS)

        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
            cat_features=cat_features,
            eval_set=(X.iloc[valid_idx], y.iloc[valid_idx]),
            early_stopping_rounds=100,
            use_best_model=True
        )

        pred = model.predict_proba(
            X.iloc[valid_idx]
        )[:, 1]

        oof_pred[valid_idx] = pred

        auc = roc_auc_score(y.iloc[valid_idx], pred)
        fold_scores.append(auc)

        print(f"Fold {fold}: AUC={auc:.6f}")

    oof_auc = roc_auc_score(y, oof_pred)

    print(f"\nMean fold AUC: {np.mean(fold_scores):.6f}")
    print(f"Overall OOF AUC: {oof_auc:.6f}")
    print(f"Delta vs baseline: {oof_auc - BASELINE_AUC:+.6f}")

    return {
        "oof_auc": oof_auc,
        "fold_scores": fold_scores,
        "oof_pred": oof_pred
    }

## Experiment E01

During EDA, we noticed that fraud rate was higher when both `velocity_score` and `merchant_risk_score` were high.

By multiplying these two features we can get new feature `risk_velocity` and test whether it improves our baseline ROC-AUC.

In [5]:
experiment_df = train_df.copy()

experiment_df["risk_velocity"] = (
    experiment_df["merchant_risk_score"]
    * experiment_df["velocity_score"]
)

experiment_features = base_features + ["risk_velocity"]

result_e01 = evaluate_features(
    experiment_df,
    experiment_features
)

Fold 1: AUC=0.943826
Fold 2: AUC=0.947445
Fold 3: AUC=0.950281
Fold 4: AUC=0.945978
Fold 5: AUC=0.941239

Mean fold AUC: 0.945754
Overall OOF AUC: 0.945692
Delta vs baseline: -0.000402


## Findings

Adding new feature `risk_velocity` slightly decreased overall `OOF ROC-AUC`, I believe that CatBoost already account the interaction between those features, so explicitly multiplying them does not seem to provide additional value.

## Experiment E02

During EDA, we observed higher fraud rates when transaction counts were high and the time since the last transaction was short.

We will create a new feature `txn_frequency` to capture this pattern and check whether it improves ROC-AUC.

In [6]:
experiment_df = train_df.copy()

experiment_df["txn_frequency"] = (
    experiment_df["txn_count_last_24h"]
    / (experiment_df["hours_since_last_txn"] + 1)
)

experiment_features = base_features + ["txn_frequency"]

result_e02 = evaluate_features(
    experiment_df,
    experiment_features
)

Fold 1: AUC=0.944645
Fold 2: AUC=0.947374
Fold 3: AUC=0.950070
Fold 4: AUC=0.946018
Fold 5: AUC=0.941529

Mean fold AUC: 0.945927
Overall OOF AUC: 0.945868
Delta vs baseline: -0.000226


## Findings

Adding `txn_frequency` slightly decreased `OOF ROC-AUC` from ***0.946094*** to ***0.945868***.

`CatBoost` may already capture the relationship between transaction count and time since the last transaction.

We will not keep this feature for now.

## Experiment E03

During EDA, we found that `amount_usd` was more informative than `account_balance_usd` when considered individually.

We will test whether the ratio between transaction amount and account balance provides additional information to CatBoost.

In [ ]:
experiment_df = train_df.copy()

experiment_df["amount_to_balance_ratio"] = (
    experiment_df["amount_usd"]
    / experiment_df["account_balance_usd"].replace(0, np.nan)
)

experiment_features = base_features + ["amount_to_balance_ratio"]

result_e03 = evaluate_features(
    experiment_df,
    experiment_features
)

Fold 1: AUC=0.943966
Fold 2: AUC=0.947544
Fold 3: AUC=0.950577
Fold 4: AUC=0.946020
Fold 5: AUC=0.941779

Mean fold AUC: 0.945977
Overall OOF AUC: 0.945924
Delta vs baseline: -0.000170


## Findings

Adding `amount_to_balance_ratio` slightly decreased `OOF ROC-AUC` from ***0.946094*** to ***0.945924***.

Although this is our best feature engineering experiment so far, the ratio does not improve the baseline.

We will not keep this feature for now.

## Experiment E04a

During EDA, we observed that transactions without authentication had a noticeably higher fraud rate.

We will create a binary feature `no_authentication` to highlight these transactions and check whether it improves `ROC-AUC`. ***Experiment E04a*** will add this feature alongside existing one. Next ***Experiment E04b*** will additionaly remove existing `auth_method` type feature.

In [8]:
experiment_df = train_df.copy()

experiment_df["no_authentication"] = (
    experiment_df["auth_method"] == "No Authentication"
).astype(int)

experiment_features = base_features + ["no_authentication"]

result_e04 = evaluate_features(
    experiment_df,
    experiment_features
)

Fold 1: AUC=0.944443
Fold 2: AUC=0.947276
Fold 3: AUC=0.950786
Fold 4: AUC=0.946069
Fold 5: AUC=0.942079

Mean fold AUC: 0.946131
Overall OOF AUC: 0.946070
Delta vs baseline: -0.000024


## Experiment E04a

As mentioned above, this experiment will additionaly remove existing `auth_method`

In [9]:
experiment_df = train_df.copy()

experiment_df["no_authentication"] = (
    experiment_df["auth_method"] == "No Authentication"
).astype(int)

experiment_features = [
    col for col in base_features
    if col != "auth_method"
] + ["no_authentication"]

result_e04b = evaluate_features(
    experiment_df,
    experiment_features
)

Fold 1: AUC=0.942305
Fold 2: AUC=0.946082
Fold 3: AUC=0.949088
Fold 4: AUC=0.944983
Fold 5: AUC=0.939840

Mean fold AUC: 0.944460
Overall OOF AUC: 0.944416
Delta vs baseline: -0.001678


## Findings

Adding `no_authentication` alongside `auth_method` produced almost the same `ROC-AUC` as the baseline (***0.946070*** vs. ***0.946094***).

However, replacing `auth_method` with the binary feature significantly reduced performance to ***0.944416***.

This suggests that the differences between authentication methods contain useful information that is lost when we convert them into a simple **True/False** feature.

We will keep the original `auth_method` feature.

## Experiment E05

During EDA, we observed very similar fraud rates across different device types. The feature also had relatively low importance in our baseline model.

We will remove `device_type` and check whether it affects `ROC-AUC`.

In [10]:
experiment_features = [
    col for col in base_features
    if col != "device_type"
]

result_e05 = evaluate_features(
    train_df,
    experiment_features
)

Fold 1: AUC=0.944084
Fold 2: AUC=0.947204
Fold 3: AUC=0.950510
Fold 4: AUC=0.946276
Fold 5: AUC=0.941647

Mean fold AUC: 0.945944
Overall OOF AUC: 0.945873
Delta vs baseline: -0.000221


## Findings

Removing `device_type` slightly decreased `OOF ROC-AUC` from ***0.946094*** to ***0.945873***.

Although this feature showed little variation in fraud rates during EDA, removing it did not improve model performance.

## Experiment E06

During EDA, `account_balance_usd` showed very limited individual predictive power, with a univariate `ROC-AUC` close to ***0.5***.

We will remove this feature and check whether it affects model performance.

In [11]:
experiment_features = [
    col for col in base_features
    if col != "account_balance_usd"
]

result_e06 = evaluate_features(
    train_df,
    experiment_features
)

Fold 1: AUC=0.944643
Fold 2: AUC=0.947521
Fold 3: AUC=0.950527
Fold 4: AUC=0.946219
Fold 5: AUC=0.942278

Mean fold AUC: 0.946238
Overall OOF AUC: 0.946140
Delta vs baseline: +0.000046


## Findings

Removing `account_balance_usd` slightly improved `OOF ROC-AUC` from ***0.946094*** to ***0.946140***.

However, the improvement is very small and may be due to random variation. Concesuently we cannot conclude that removing `account_balance_usd` consistently improves model performance.

## Summary and Next Steps

We tested several feature engineering and feature selection ideas based on our EDA findings.

Most experiments slightly decreased ROC-AUC, suggesting that CatBoost may already capture much of the information contained in these simple feature transformations.

Removing `account_balance_usd` produced the best result (**0.946140**), but the improvement over our baseline (**0.946094**) is too small to draw a strong conclusion.

Instead of continuing to add or remove features, we will now focus on understanding where our model makes mistakes.

**Next step:** Analyze OOF predictions to identify difficult transactions and potential weaknesses in the model.